# 2026 Free-Agent Fit Model

This notebook loads 2026 NBA free-agent information, merges it with the existing 2025-26 player stats dataset, and applies the same first-pass Bulls fit score from `phase2_fresh.ipynb`.

The goal is to move from "which players fit statistically?" to "which available free agents fit statistically and financially?"

## Data Source

Free-agent status comes from Spotrac's 2026 NBA free-agent table. The notebook saves a local CSV snapshot in `data/nba_free_agents_2026_spotrac.csv` after loading it, so the analysis can still be inspected later without repeatedly scraping the site.

Important modeling choice: the default `available_free_agents` table includes `UFA` and `RFA` players. Player-option and club-option players are loaded into the raw table, but excluded from the default available filter because they are conditional rather than guaranteed market options.

In [ ]:
from io import StringIO
from pathlib import Path
import re
import unicodedata
import warnings

import pandas as pd
import requests

try:
    from bs4 import BeautifulSoup
except ImportError:
    BeautifulSoup = None

project_dir = Path.cwd()
if not (project_dir / "data").exists():
    project_dir = project_dir / "fresh_start"

data_dir = project_dir / "data"
stats_path = data_dir / "nba_all_players_2526.csv"
free_agent_snapshot_path = data_dir / "nba_free_agents_2026_spotrac.csv"

league_df = pd.read_csv(stats_path)
league_df.shape

## Load 2026 Free Agents

This cell tries to load fresh data from Spotrac first. If that fails, it falls back to the local CSV snapshot if one exists. This is useful because web pages can change, internet can be unavailable, or certificate settings can differ across machines.

In [ ]:
SPOTRAC_FREE_AGENTS_URL = "https://www.spotrac.com/nba/free-agents/_/year/2026"

def parse_html_table_with_bs4(html):
    if BeautifulSoup is None:
        raise ImportError("BeautifulSoup is not installed in this Python environment.")

    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table")
    if table is None:
        raise ValueError("No HTML table found on Spotrac page.")

    headers = [cell.get_text(" ", strip=True) for cell in table.find_all("th")]
    rows = []
    for row in table.find_all("tr"):
        cells = [cell.get_text(" ", strip=True) for cell in row.find_all("td")]
        if cells:
            rows.append(cells)

    return pd.DataFrame(rows, columns=headers[: len(rows[0])])

def load_spotrac_free_agents(url=SPOTRAC_FREE_AGENTS_URL):
    headers = {"User-Agent": "Mozilla/5.0"}
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        response = requests.get(url, headers=headers, timeout=30, verify=False)
    response.raise_for_status()

    try:
        tables = pd.read_html(StringIO(response.text))
        if tables:
            return tables[0]
    except ImportError:
        pass

    return parse_html_table_with_bs4(response.text)

try:
    raw_free_agents = load_spotrac_free_agents()
    source_used = "Spotrac live page"
    raw_free_agents.to_csv(free_agent_snapshot_path, index=False)
except Exception as error:
    if free_agent_snapshot_path.exists():
        raw_free_agents = pd.read_csv(free_agent_snapshot_path)
        source_used = f"local snapshot after live load failed: {error}"
    else:
        raise

print(source_used)
print(raw_free_agents.shape)
raw_free_agents.head(10)

## Clean Free-Agent Fields

The source table has salary strings and combined free-agent type labels like `UFA / Bird` or `PLAYER / $49.0M`. This step turns those into cleaner columns that are easier to filter.

In [ ]:
def clean_money(value):
    if pd.isna(value):
        return pd.NA
    cleaned = re.sub(r"[^0-9.]", "", str(value))
    return float(cleaned) if cleaned else pd.NA

def normalize_player_name(name):
    if pd.isna(name):
        return ""
    name = str(name).lower()
    name = unicodedata.normalize("NFKD", name)
    name = "".join(char for char in name if not unicodedata.combining(char))
    name = name.replace(".", "")
    name = name.replace("'", "")
    name = name.replace("’", "")
    name = name.replace("‘", "")
    name = name.replace("`", "")
    name = re.sub(r"[^a-z0-9 ]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    name = re.sub(r"\b(jr|sr|ii|iii|iv)\b", "", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name

free_agents = raw_free_agents.copy()
free_agents = free_agents.rename(columns={free_agents.columns[0]: "PLAYER_NAME"})
free_agents.columns = [
    str(col).strip().upper().replace(" ", "_").replace("(", "").replace(")", "")
    for col in free_agents.columns
]
free_agents = free_agents.rename(columns={
    "PLAYER_238": "PLAYER_NAME",
    "PREV_TEAM": "PREV_TEAM",
    "PREV_AAV": "PREV_AAV",
})

free_agents["PREV_AAV_CLEAN"] = free_agents["PREV_AAV"].apply(clean_money)
free_agents["FA_CLASS"] = free_agents["TYPE"].str.split("/").str[0].str.strip()
free_agents["RIGHTS_OR_OPTION"] = free_agents["TYPE"].str.split("/").str[1].str.strip()
free_agents["IS_DEFAULT_AVAILABLE"] = free_agents["FA_CLASS"].isin(["UFA", "RFA"])
free_agents["IS_UNRESTRICTED"] = free_agents["FA_CLASS"].eq("UFA")
free_agents["IS_RESTRICTED"] = free_agents["FA_CLASS"].eq("RFA")
name_aliases = {
    "nahshon hyland": "bones hyland",
    "cameron thomas": "cam thomas",
}

free_agents["NAME_KEY"] = free_agents["PLAYER_NAME"].apply(normalize_player_name)
free_agents["NAME_KEY"] = free_agents["NAME_KEY"].replace(name_aliases)

free_agents[["PLAYER_NAME", "POS", "AGE", "YOE", "PREV_TEAM", "PREV_AAV", "PREV_AAV_CLEAN", "FA_CLASS", "RIGHTS_OR_OPTION", "IS_DEFAULT_AVAILABLE"]].head(15)

## Merge Free Agents With Player Stats

The fit score needs basketball production stats, so this step joins the free-agent table to `nba_all_players_2526.csv` using a normalized player-name key. Any unmatched players should be reviewed manually, because name formatting can differ across sources.

In [ ]:
league_stats = league_df.copy()
league_stats["NAME_KEY"] = league_stats["PLAYER_NAME"].apply(normalize_player_name)

free_agent_stats = free_agents.merge(
    league_stats,
    on="NAME_KEY",
    how="left",
    suffixes=("_FA", "_STATS"),
)

matched_count = free_agent_stats["PLAYER_ID"].notna().sum()
print(f"Matched {matched_count:,} of {len(free_agent_stats):,} free agents to the stats file.")

unmatched_free_agents = (
    free_agent_stats[free_agent_stats["PLAYER_ID"].isna()]
    [["PLAYER_NAME_FA", "POS", "AGE_FA", "PREV_TEAM", "TYPE", "PREV_AAV"]]
    .reset_index(drop=True)
)

unmatched_free_agents.head(25)

## Apply The Bulls Fit Score

This uses the same first-pass score from the Phase 2 notebook. The weights emphasize defensive impact, rebounding, and frontcourt help, while still rewarding shooting, scoring, and plus-minus. Age receives a small negative weight to favor players who better fit a retool/rebuild timeline.

In [ ]:
score_features = {
    "BLK": 0.22,
    "DREB": 0.18,
    "REB": 0.12,
    "STL": 0.12,
    "FG3_PCT": 0.10,
    "FG3M": 0.08,
    "PTS": 0.08,
    "PLUS_MINUS": 0.06,
    "AGE_STATS": -0.04,
}

scored_free_agents = free_agent_stats[free_agent_stats["PLAYER_ID"].notna()].copy()

for feature, weight in score_features.items():
    mean = scored_free_agents[feature].mean()
    std = scored_free_agents[feature].std(ddof=0)
    scored_free_agents[f"{feature}_z"] = (scored_free_agents[feature] - mean) / std if std else 0

scored_free_agents["fit_score"] = sum(
    scored_free_agents[f"{feature}_z"] * weight
    for feature, weight in score_features.items()
)

scored_free_agents.shape

## Filter To Available Free Agents

The default table keeps unrestricted and restricted free agents. Restricted free agents are less available than unrestricted free agents because their current team can match an offer, so use `IS_UNRESTRICTED` when you want the cleanest market-only list.

In [ ]:
MAX_PREV_AAV = 35_000_000
MIN_GAMES_PLAYED = 20

target_columns = [
    "PLAYER_NAME_FA", "POS", "AGE_FA", "YOE", "PREV_TEAM", "FA_CLASS", "RIGHTS_OR_OPTION",
    "PREV_AAV_CLEAN", "GP", "MIN", "PTS", "AST", "REB", "DREB", "BLK", "STL", "FG3_PCT", "FG3M",
    "PLUS_MINUS", "fit_score"
]

available_free_agents = (
    scored_free_agents[
        scored_free_agents["IS_DEFAULT_AVAILABLE"]
        & (scored_free_agents["TEAM_ABBREVIATION"] != "CHI")
        & (scored_free_agents["GP"] >= MIN_GAMES_PLAYED)
        & (scored_free_agents["PREV_AAV_CLEAN"] <= MAX_PREV_AAV)
    ]
    .sort_values("fit_score", ascending=False)
    [target_columns]
    .reset_index(drop=True)
)

available_free_agents.head(30)

## Cleaner Market: Unrestricted Free Agents Only

This version excludes RFAs. It is less exciting, but usually more realistic because the Bulls would not need another team to decline matching rights.

In [ ]:
unrestricted_targets = (
    available_free_agents[available_free_agents["FA_CLASS"] == "UFA"]
    .reset_index(drop=True)
)

unrestricted_targets.head(30)

## Build A Free-Agent Decision Board

The fit score ranks statistical fit, but the next step is basketball interpretation. We will add two simple columns:

1. `role_label`: what problem the player might solve for Chicago.
2. `target_tier`: how realistic and useful the target looks after considering fit, free-agent type, and salary.

This is where the project starts becoming a decision tool instead of only a ranking table.

### Step 1: Label Player Roles

These rules are intentionally simple. A player can only receive one role label here, so the order matters. For example, a center with strong blocks and rebounds becomes a `rim-protecting big` before we check whether he is also a general frontcourt depth piece.

In [ ]:
def assign_role_label(row):
    pos = str(row["POS"])

    if pos == "C" and row["BLK"] >= 1.0 and row["REB"] >= 6.0:
        return "rim-protecting big"
    if pos in ["PF", "C"] and row["REB"] >= 6.0:
        return "frontcourt rebounder"
    if pos in ["SF", "PF", "SG"] and row["STL"] >= 0.9 and row["REB"] >= 4.0:
        return "wing defender"
    if row["FG3_PCT"] >= 0.37 and row["FG3M"] >= 1.5:
        return "floor spacer"
    if pos in ["PG", "SG", "G"] and row["AST"] >= 3.0:
        return "guard creator"
    if row["PTS"] >= 12.0:
        return "secondary scorer"
    return "depth option"

decision_board = available_free_agents.copy()
decision_board["role_label"] = decision_board.apply(assign_role_label, axis=1)

decision_board[
    ["PLAYER_NAME_FA", "POS", "AGE_FA", "FA_CLASS", "PREV_AAV_CLEAN", "fit_score", "role_label"]
].head(20)

### Step 2: Add Target Tiers

The tier is not meant to be final truth. It is a first-pass filter that helps you decide which names deserve research. In general, unrestricted free agents are easier to pursue than restricted free agents, cheaper players are easier to fit under the cap, and strong fit-score players deserve more attention.

In [ ]:
def assign_target_tier(row):
    salary = row["PREV_AAV_CLEAN"]
    fit = row["fit_score"]
    fa_class = row["FA_CLASS"]

    if fa_class == "UFA" and fit >= 0.75 and salary <= 15_000_000:
        return "priority realistic target"
    if fa_class == "UFA" and fit >= 0.75 and salary <= 30_000_000:
        return "priority expensive target"
    if fa_class == "RFA" and fit >= 1.00:
        return "high fit but difficult"
    if salary > 25_000_000:
        return "expensive / low flexibility"
    if fit >= 0.40:
        return "secondary target"
    return "low-priority depth"

decision_board["target_tier"] = decision_board.apply(assign_target_tier, axis=1)

tier_order = {
    "priority realistic target": 1,
    "priority expensive target": 2,
    "high fit but difficult": 3,
    "secondary target": 4,
    "expensive / low flexibility": 5,
    "low-priority depth": 6,
}

decision_board["target_tier_rank"] = decision_board["target_tier"].map(tier_order)
decision_board = decision_board.sort_values(
    ["target_tier_rank", "fit_score"],
    ascending=[True, False],
).reset_index(drop=True)

decision_board[
    [
        "PLAYER_NAME_FA", "POS", "AGE_FA", "FA_CLASS", "PREV_TEAM",
        "PREV_AAV_CLEAN", "fit_score", "role_label", "target_tier"
    ]
].head(40)

### Step 3: Review By Role And Tier

This summary helps check whether the model is solving the Bulls' actual problems. If most of the best options are guards, but the project diagnosis says the Bulls need frontcourt defense, then the weights or filters need adjustment.

In [ ]:
role_summary = (
    decision_board
    .groupby(["role_label", "target_tier"])
    .agg(
        player_count=("PLAYER_NAME_FA", "count"),
        avg_fit_score=("fit_score", "mean"),
        median_prev_aav=("PREV_AAV_CLEAN", "median"),
    )
    .sort_values(["avg_fit_score", "player_count"], ascending=[False, False])
    .reset_index()
)

role_summary

### Step 4: Create A Research Shortlist

This is the table to research manually. For each player, the next question is not just whether the number looks good, but whether the player is obtainable, healthy, affordable, and a clean fit next to the Bulls' young core.

In [ ]:
research_shortlist = (
    decision_board[
        decision_board["target_tier"].isin([
            "priority realistic target",
            "priority expensive target",
            "high fit but difficult",
            "secondary target",
        ])
    ]
    .sort_values("fit_score", ascending=False)
    .reset_index(drop=True)
)

research_shortlist[
    [
        "PLAYER_NAME_FA", "POS", "AGE_FA", "FA_CLASS", "PREV_TEAM",
        "PREV_AAV_CLEAN", "GP", "MIN", "PTS", "REB", "BLK", "STL",
        "FG3_PCT", "fit_score", "role_label", "target_tier"
    ]
].head(25)

## Next Model Improvements

- Add a projected salary column instead of using previous AAV as the cost proxy.
- Add position constraints so the model does not recommend too many guards or centers.
- Split scoring into `retool_score` and `rebuild_score`.
- Add a manual `realistic_target` flag after reviewing contract situation, age, injury risk, and likely market demand.
- Compare free-agent targets against draft targets so the Bulls do not solve the same roster need twice.